# Ingesta de datos de precipitaciones mensuales promedio de CONAGUA
Esta libreta de Jupyter se utilizará para la ingesta y procesamiento de datos relacionados al registro de precipitaciones  medias en distintas estaciones metereológicas distribuidas en todo México. Los datos provienen de la base de datos de la CONAGUA. 

## Ingesta de Datos

In [ ]:
# Importacion de librerias
import os # Para crear las carpetas y construir las rutas de los archivos en el disco.
import requests # Para realizar la conexión HTTP y descargar los archivos binarios.
import urllib3 # Para deshabilitar los avisos de certificados SSL de páginas de gobierno.
import pandas as pd
import glob

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) # Desactivar avisos de certificados SSL no verificados.

# Definicion de la URL de la base de datos de lluvia del SMN (Servicio Meteorológico Nacional) de CONAGUA.
base_url = "https://smn.conagua.gob.mx/tools/RESOURCES/com_archivo_datos_resumenes"

# Ruta para guardar en data/raw/ subiendo un nivel desde la carpeta notebooks/
# Usamos '..' para salir de notebooks/ hacia la raíz del proyecto
main_folder = os.path.join("..", "data", "raw", "datos_lluvia_2019_2023")
os.makedirs(main_folder, exist_ok=True)

# Cabecera HTTP estándar para simular una petición de navegador.
headers = {"User-Agent": "Mozilla/5.0"}

# Definimos el rango de años: range(2019, 2024) abarca desde 2019 hasta 2023.
start_year = 2019
end_year = 2024

# Bucle for para recorrer cada año en el rango definido y descargar los archivos de lluvia correspondientes a cada mes.
for current_year in range(start_year, end_year):
    year_str = str(current_year)
    print("----------------------------------------")
    print("Descarga del año: " + year_str)
    print("----------------------------------------")
    
    # Creamos una subcarpeta para cada año (ej. ../data/raw/datos_lluvia_2019_2023/lluvia_2019)
    year_folder = os.path.join(main_folder, "lluvia_" + year_str)
    os.makedirs(year_folder, exist_ok=True)
    
    # Recorremos los meses del 1 al 12 (enero a diciembre)
    for month in range(1, 13):
        mes_str = str(month).zfill(2) 
        file_name = year_str + mes_str + "010000Lluv.csv" # Formato: [AÑO] + [MES] + 010000Lluv.csv
        file_url = base_url + "/" + file_name # Url completa del archivo en el servidor
        local_path = os.path.join(year_folder, file_name) # Ruta local final del archivo
        
        try:
            res = requests.get(file_url, headers=headers, verify=False) # Peticion GET al servidor

            if res.status_code == 200: 
                with open(local_path, "wb") as f:
                    f.write(res.content)
                print("Ano " + year_str + " | Mes " + mes_str + ": Descargado con exito")
            else:
                # Si el mes no está disponible en el servidor (ej. código 404)
                print("Ano " + year_str + " | Mes " + mes_str + ": No disponible (Status " + str(res.status_code) + ")")
                
        except Exception as e:
            # Por si se interrumpe la conexión a internet
            print("Error al descargar " + file_name + ": " + str(e))

print("\nDescargas de 2019 a 2023 completadas.")


----------------------------------------
Descarga del año: 2019
----------------------------------------
Ano 2019 | Mes 01: Descargado con exito
Ano 2019 | Mes 02: Descargado con exito
Ano 2019 | Mes 03: Descargado con exito
Ano 2019 | Mes 04: Descargado con exito
Ano 2019 | Mes 05: Descargado con exito
Ano 2019 | Mes 06: Descargado con exito
Ano 2019 | Mes 07: Descargado con exito
Ano 2019 | Mes 08: Descargado con exito
Ano 2019 | Mes 09: Descargado con exito
Ano 2019 | Mes 10: Descargado con exito
Ano 2019 | Mes 11: Descargado con exito
Ano 2019 | Mes 12: Descargado con exito
----------------------------------------
Descarga del año: 2020
----------------------------------------
Ano 2020 | Mes 01: Descargado con exito
Ano 2020 | Mes 02: Descargado con exito
Ano 2020 | Mes 03: Descargado con exito
Ano 2020 | Mes 04: Descargado con exito
Ano 2020 | Mes 05: Descargado con exito
Ano 2020 | Mes 06: Descargado con exito
Ano 2020 | Mes 07: Descargado con exito
Ano 2020 | Mes 08: Descargado 

## Procesamiento y limpieza

In [ ]:

# 1. Definir la ruta hacia los datos crudos desde la carpeta 'notebooks'
# Buscamos recursivamente todos los .csv dentro de las carpetas de lluvia
patron = os.path.join("..", "data", "raw", "lluvia_*", "*.csv")

# Definir rutas absolutas con glob.glob
archivos = sorted(glob.glob(patron))
print(f"Total de archivos CSV encontrados: {len(archivos)}")

lista_nacional = []
cols_fijas = ["lon", "lat", "edo", "clave", "estacion"]


# 2. Procesar cada archivo mensual manteniendo todos los estados
for ruta in archivos:
  try:
    df_mes = pd.read_csv(ruta, encoding="latin-1")

    # Limpiar y normalizar nombres de columnas a minúsculas
    df_mes.columns = [c.strip().lower() for c in df_mes.columns]

    # Identificar la columna de precipitación (la que cambia de nombre cada mes)
    col_medicion = [c for c in df_mes.columns if c not in cols_fijas]

    # En esta parte del codigo vamos a crear una columna con el periodo (el mes y a;o) donde se hizo la medicion. 
    if col_medicion:
      nombre_lluvia = col_medicion[0]
      # Guardar el periodo original (ej. 'ene-19')
      df_mes["periodo"] = nombre_lluvia
      # Estandarizar nombre a 'precipitacion_mm'
      df_mes.rename(columns={nombre_lluvia: "precipitacion_mm"}, inplace=True)

    # Conservar el archivo de origen
    df_mes["archivo_origen"] = os.path.basename(ruta)

    lista_nacional.append(df_mes)

  except Exception as e:
    print(f"Error al leer {ruta}: {e}")

    # 3. Concatenar todas las estaciones del país
if lista_nacional:
  df_lluvia_nacional = pd.concat(lista_nacional, ignore_index=True)
  print(
      f"¡Dataset nacional consolidado! Filas: {len(df_lluvia_nacional):,} |"
      f" Columnas: {df_lluvia_nacional.shape[1]}"
  )
  display(df_lluvia_nacional.head())

  # limpieza y tipificación de temporal del dataframe 

# Creacion de columnas Fecha, Año, y Mes
df_lluvia_nacional['anio'] = (df_lluvia_nacional['archivo_origen'].str[:4].astype(int)) # Crear columna con el año utilizando los primeros 4 digitos del nombre de archivo de origen
df_lluvia_nacional['mes'] = (df_lluvia_nacional['archivo_origen'].str[4:6].astype(int)) # Crear columna con el año utilizando los siguientes dos digitos del nombre de archivo de origen

# Crear fecha
anio = df_lluvia_nacional['anio'].astype(str)
mes = df_lluvia_nacional['mes'].astype(str)
df_lluvia_nacional['fecha'] = pd.to_datetime(anio + "-" + mes + '-01')

df_lluvia_nacional.head()

# Guardar el archivo consolidado
carpeta_salida = os.path.join('..', 'data', 'processed')
os.makedirs(carpeta_salida, exist_ok=True)

# Ruta de salida
ruta_salida = os.path.join("..", "data", "processed", 'lluvia_nacional_2019_2023.csv')

# Crear df en la ruta de salida
df_lluvia_nacional.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

Total de archivos CSV encontrados: 60
¡Dataset nacional consolidado! Filas: 75,695 | Columnas: 8


,lon,lat,edo,clave,estacion,precipitacion_mm,periodo,archivo_origen
0,-102.309722,21.895000,AGS,AGSAG,"Aguascalientes, Ags.",6.60,ene-19,201901010000Lluv.csv
1,-102.585833,22.177222,AGS,ALMAG,"Alamitos, Ags.",11.50,ene-19,201901010000Lluv.csv
2,-102.464167,22.188611,AGS,ANVAG,"Cincuenta Aniversario, Ags.",10.60,ene-19,201901010000Lluv.csv
3,-102.184167,21.738611,AGS,BRTAG,"San Bartolo, Ags.",10.50,ene-19,201901010000Lluv.csv
4,-102.712222,21.849167,AGS,CALVILLO,"Calvillo, Ags. SMN*",18.17,ene-19,201901010000Lluv.csv


# Validación de datos 
### Schema basándonos en el diccionario de datos

In [9]:
from datetime import date, datetime
from typing import Optional
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, field_validator


class LluviaEstacionSchema(BaseModel):
  model_config = ConfigDict(extra="ignore")

  # 1. Variables de ubicación geográfica
  lon: Optional[float] = Field(default=None, ge=-180.0, le=180.0)
  lat: Optional[float] = Field(default=None, ge=-90.0, le=90.0)

  # 2. Identificación de la estación y entidad
  clave: Optional[str] = None
  edo: Optional[str] = None
  estacion: Optional[str] = None

  # 3. Medición meteorológica (Precipitación acumulada mensual en mm)
  # ge=0.0 asegura que la lámina de lluvia no sea negativa
  precipitacion_mm: Optional[float] = Field(
      default=None, ge=0.0, le=3000.0
  )  # Límite superior holgado mensual

  # 4. Metadatos de origen
  periodo: Optional[str] = None
  archivo_origen: Optional[str] = None

  # 5. Variables cronológicas derivadas
  anio: Optional[int] = Field(default=None, ge=1900, le=2100)
  mes: Optional[int] = Field(default=None, ge=1, le=12)
  fecha: Optional[date] = None

  # Validador 1: Tratamiento de nulos y códigos centinela del SMN/CONAGUA (-99.9, -999)
  @field_validator("*", mode="before")
  @classmethod
  def normalize_missing_and_sentinels(cls, value):
    if value is None or pd.isna(value):
      return None

    # Limpiar cadenas vacías o textos como 'nan'
    if str(value).strip().lower() in ["", "nan", "none", "null"]:
      return None

    # Detectar códigos numéricos centinela (ausencia de dato / estación inactiva)
    try:
      val_num = float(value)
      if val_num in [-99.9, -999.0, -99.0]:
        return None
    except (ValueError, TypeError):
      pass

    return value

  # Validador 2: Estandarización de campos de texto (eliminar espacios accidentales)
  @field_validator(
      "clave", "edo", "estacion", "periodo", "archivo_origen", mode="before"
  )
  @classmethod
  def clean_text_fields(cls, value):
    if value is None or pd.isna(value):
      return None
    return str(value).strip()

  # Validador 3: Conversión de formatos temporales a objeto date
  @field_validator("fecha", mode="before")
  @classmethod
  def parse_dates(cls, value):
    if value is None or pd.isna(value):
      return None
    if isinstance(value, (pd.Timestamp, datetime)):
      return value.date()
    if isinstance(value, str):
      return datetime.strptime(value.strip()[:10], "%Y-%m-%d").date()
    return value

### Función para validar los datos descargados

In [10]:
from pydantic import ValidationError

def validate_rain_df(df: pd.DataFrame) -> pd.DataFrame:
  validated_rows = []

  for index, row in df.iterrows():
    try:
      rain_data = LluviaEstacionSchema(**row.to_dict())
      validated_rows.append(rain_data.model_dump())

    except ValidationError as e:
      print(f"Error en fila {index}:")
      print(e)

  return pd.DataFrame(validated_rows)

In [11]:
#Validar el DataFrame consolidado de lluvia
df_lluvia_validada = validate_rain_df(df_lluvia_nacional)

#Inspeccionar el resultado
print(f"Filas originales: {len(df_lluvia_nacional):,}")
print(f"Filas validadas:  {len(df_lluvia_validada):,}")
display(df_lluvia_validada.head(5))

Filas originales: 75,695
Filas validadas:  75,695


,lon,lat,clave,edo,estacion,precipitacion_mm,periodo,archivo_origen,anio,mes,fecha
0,-102.309722,21.895000,AGSAG,AGS,"Aguascalientes, Ags.",6.60,ene-19,201901010000Lluv.csv,2019,1,2019-01-01
1,-102.585833,22.177222,ALMAG,AGS,"Alamitos, Ags.",11.50,ene-19,201901010000Lluv.csv,2019,1,2019-01-01
2,-102.464167,22.188611,ANVAG,AGS,"Cincuenta Aniversario, Ags.",10.60,ene-19,201901010000Lluv.csv,2019,1,2019-01-01
3,-102.184167,21.738611,BRTAG,AGS,"San Bartolo, Ags.",10.50,ene-19,201901010000Lluv.csv,2019,1,2019-01-01
4,-102.712222,21.849167,CALVILLO,AGS,"Calvillo, Ags. SMN*",18.17,ene-19,201901010000Lluv.csv,2019,1,2019-01-01


In [ ]:
# Guardar el archivo final
ruta_salida = os.path.join(
    "..", "data", "processed", "lluvia_nacional_2019_2023.csv"
)
df_lluvia_validada.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo exportado exitosamente a: {ruta_salida}")

Archivo exportado exitosamente a: ..\data\processed\lluvia_nacional_2019_2023.csv
